# Week 2 Reference: Static & Dynamic Arrays, Binary Search, Simple Sorts

**Z5007 Programming and Data Structures (MTech DSAI Core) · IIT Madras Zanzibar**

Companion runnable reference implementations for Lab 1.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5007/week-02/labs/W02_Reference_Arrays_Sorts.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every algorithm is implemented from first principles in Python and verified with assertions.

## 1. Static Arrays, Binary Search, and Lower Bound

Contiguous arrays provide $O(1)$ indexing and cache locality, whereas shifting elements on middle insertion or deletion costs $O(n)$. Sorted arrays support $O(\log n)$ binary search and `lower_bound` queries.

In [1]:
def binary_search(arr, target):
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == target:
            return mid
        if arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1


def lower_bound(arr, target):
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    return lo


xs = [3, 7, 7, 7, 12, 19, 25]
assert binary_search(xs, 19) == 5
assert binary_search(xs, 10) == -1
assert lower_bound(xs, 7) == 1
assert lower_bound(xs, 8) == 4
print("Binary search and lower_bound checks passed.")


Binary search and lower_bound checks passed.


## 2. Dynamic Arrays & Amortized Doubling

Doubling capacity when full copies at most $1 + 2 + 4 + \dots + n < 2n$ items across $n$ appends, yielding $O(1)$ amortized cost per `append`. Shrinking by half only at $1/4$ occupancy prevents thrashing around a boundary.

In [2]:
class DynamicArray:
    def __init__(self, initial_capacity=2):
        self._data = [None] * initial_capacity
        self._size = 0
        self.total_copies = 0

    def __len__(self):
        return self._size

    @property
    def capacity(self):
        return len(self._data)

    def _resize(self, new_cap):
        block = [None] * new_cap
        for i in range(self._size):
            block[i] = self._data[i]
            self.total_copies += 1
        self._data = block

    def append(self, x):
        if self._size == len(self._data):
            self._resize(2 * len(self._data))
        self._data[self._size] = x
        self._size += 1

    def pop(self):
        if self._size == 0:
            raise IndexError("empty")
        val = self._data[self._size - 1]
        self._data[self._size - 1] = None
        self._size -= 1
        if 0 < self._size <= len(self._data) // 4 and len(self._data) > 2:
            self._resize(len(self._data) // 2)
        return val


da = DynamicArray(2)
for i in range(1, 257):
    da.append(i)
print(f"After 256 appends: size={len(da)}, capacity={da.capacity}, total_copies={da.total_copies}")
assert da.total_copies < 2 * 256


After 256 appends: size=256, capacity=256, total_copies=254


## 3. Bubble Sort & Selection Sort with Operation Counters

Bubble sort stops early in $O(n)$ when the input is already sorted. Selection sort always inspects every pair in the suffix ($n(n-1)/2$ comparisons) yet performs at most $n - 1$ swaps.

In [3]:
import random


def bubble_sort(arr):
    comps = swaps = 0
    for end in range(len(arr) - 1, 0, -1):
        moved = False
        for i in range(end):
            comps += 1
            if arr[i] > arr[i + 1]:
                arr[i], arr[i + 1] = arr[i + 1], arr[i]
                swaps += 1
                moved = True
        if not moved:
            break
    return comps, swaps


def selection_sort(arr):
    comps = swaps = 0
    n = len(arr)
    for i in range(n - 1):
        m = i
        for j in range(i + 1, n):
            comps += 1
            if arr[j] < arr[m]:
                m = j
        if m != i:
            arr[i], arr[m] = arr[m], arr[i]
            swaps += 1
    return comps, swaps


rng = random.Random(2)
test_data = [rng.randint(0, 100) for _ in range(40)]
a, b = test_data[:], test_data[:]
c_b, s_b = bubble_sort(a)
c_s, s_s = selection_sort(b)
assert a == sorted(test_data) and b == sorted(test_data)
print(f"n=40 random -> bubble: {c_b} comps, {s_b} swaps | selection: {c_s} comps, {s_s} swaps")


n=40 random -> bubble: 780 comps, 416 swaps | selection: 780 comps, 38 swaps
